# False Positive (Hallucination) Error Analysis

Qualitative error analysis for LLM over-prediction of persuasion techniques.

Two-axis structure:
1. **Per-label FP counts** — which techniques are most hallucinated across all models
2. **Consensus errors** — examples where ≥N models hallucinate the same label (systemic)
3. **Model variance** — labels where error rate differs strongly across models (model-specific)

In [ ]:
import json
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 50)

# --- config ---
OUTPUTS_DIR = Path("../outputs")
PROMPT      = "NEW_Prompt"
SPLIT       = "test"   # "test" or "dev"
N_CONSENSUS = 6

# derived from SPLIT — do not edit
FOLDER_PREFIX = "test_" if SPLIT == "test" else "dev_"
SPLIT_SUBDIR  = "test"  if SPLIT == "test" else "development"

MODEL_SHORT = {

    "test_gpt-4.1-2025-04-14":                "GPT-4.1",
    "test_gpt-4.1-mini-2025-04-14":           "GPT-4.1-mini",
    "test_gpt-5-mini-2025-08-07":             "GPT-5-mini",
    "test_gpt-5.2-2025-12-11":                "GPT-5.2",
    "test_Qwen_Qwen2.5-72B-Instruct":         "Qwen-72B",
    "test_meta-llama_Llama-3.3-70B-Instruct": "Llama-70B",

    "dev_gpt-4.1-2025-04-14":                "GPT-4.1",
    "dev_gpt-4.1-mini-2025-04-14":           "GPT-4.1-mini",
    "dev_gpt-5-mini-2025-08-07":             "GPT-5-mini",
    "dev_gpt-5.2-2025-12-11":                "GPT-5.2",
    "dev_Qwen_Qwen2.5-72B-Instruct":         "Qwen-72B",
    "dev_meta-llama_Llama-3.3-70B-Instruct": "Llama-70B",
}


## 1. Load predictions

In [ ]:
def load_model_predictions(prompt_dir: Path, model_folder: str, split_subdir: str) -> pd.DataFrame:
    records = []
    model_path = prompt_dir / model_folder
    for domain_dir in sorted(model_path.iterdir()):
        if not domain_dir.is_dir():
            continue
        pred_file = domain_dir / split_subdir / "predictions.jsonl"
        if not pred_file.exists():
            continue
        with open(pred_file) as f:
            for line in f:
                rec = json.loads(line)
                rec["domain"] = domain_dir.name
                # some older outputs lack custom_id — synthesize it
                if "custom_id" not in rec:
                    rec["custom_id"] = f"{domain_dir.name}|{split_subdir}|{rec['id']}"
                records.append(rec)
    if not records:
        raise ValueError(f"No predictions found for {model_folder} / {split_subdir}")
    df = pd.DataFrame(records)
    df["gold"] = df["gold"].apply(lambda x: x if isinstance(x, list) else [])
    df["pred"] = df["pred"].apply(lambda x: x if isinstance(x, list) else [])
    df["fp_labels"] = df.apply(lambda r: set(r["pred"]) - set(r["gold"]), axis=1)
    df["fn_labels"] = df.apply(lambda r: set(r["gold"]) - set(r["pred"]), axis=1)
    df["tp_labels"] = df.apply(lambda r: set(r["pred"]) & set(r["gold"]), axis=1)
    return df.set_index("custom_id")


prompt_dir = OUTPUTS_DIR / PROMPT
model_folders = sorted([
    d.name for d in prompt_dir.iterdir()
    if d.is_dir() and d.name.startswith(FOLDER_PREFIX)
])

dfs = {}
for folder in model_folders:
    short = MODEL_SHORT.get(folder, folder)
    dfs[short] = load_model_predictions(prompt_dir, folder, SPLIT_SUBDIR)

MODEL_NAMES = list(dfs.keys())
print("Loaded models:")
for name, df in dfs.items():
    print(f"  {name}: {len(df):,} examples")


## 2. Per-label FP counts across all models

Which persuasion techniques are hallucinated most in absolute terms?

In [ ]:
label_fp: dict[str, dict[str, int]] = defaultdict(lambda: defaultdict(int))

for model_name, df in dfs.items():
    for fp_set in df["fp_labels"]:
        for label in fp_set:
            label_fp[label][model_name] += 1

rows = []
for label, model_counts in label_fp.items():
    rows.append({"label": label, "total_fp": sum(model_counts.values()), **model_counts})

label_fp_df = (
    pd.DataFrame(rows)
    .fillna(0)
    .astype({c: int for c in ["total_fp"] + MODEL_NAMES if c in pd.DataFrame(rows).columns})
    .sort_values("total_fp", ascending=False)
    .set_index("label")
)

# all labels — there are only 7 in this task
top_labels = label_fp_df.index.tolist()

display_cols = ["total_fp"] + [m for m in MODEL_NAMES if m in label_fp_df.columns]
print("Most hallucinated labels (total FP across all models):\n")
display(label_fp_df[display_cols])

In [ ]:
# Absolute counts heatmap
heatmap_data = label_fp_df.loc[top_labels, MODEL_NAMES].astype(int)

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(
    heatmap_data,
    annot=True, fmt="d", cmap="YlOrRd",
    linewidths=0.5, ax=ax,
    cbar_kws={"label": "FP count"},
)
ax.set_title("False Positives per label per model (absolute counts)", pad=12)
ax.set_xlabel("")
ax.set_ylabel("")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Relative frequency heatmap: FP rate = FP / total examples for that model
# Reads as: "in X% of all test texts, this model hallucinated this label"
model_sizes = {name: len(df) for name, df in dfs.items()}
fp_rate_df = label_fp_df[MODEL_NAMES].copy().astype(float)
for model in MODEL_NAMES:
    fp_rate_df[model] = fp_rate_df[model] / model_sizes[model]

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(
    fp_rate_df.loc[top_labels],
    annot=True, fmt=".2f", cmap="YlOrRd",
    linewidths=0.5, ax=ax,
    cbar_kws={"label": "FP rate (FP / total examples)"},
)
ax.set_title(
    "False Positive rate per label per model\n"
    "(fraction of all test examples where the label is hallucinated)",
    pad=12,
)
ax.set_xlabel("")
ax.set_ylabel("")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 3. Consensus error analysis

For each `(example, hallucinated_label)` pair, count how many models predict the label when it is not in the gold annotation.  
Cases where **≥ N_CONSENSUS** models agree reveal *systemic* confusion — the text itself triggers the error, not a model-specific quirk.

In [ ]:
# Inner join: only examples present in every model's output
common_ids = set.intersection(*[set(df.index) for df in dfs.values()])
print(f"Examples present in all {len(dfs)} models: {len(common_ids):,}")

consensus_raw: dict[tuple, dict] = defaultdict(lambda: {"models": [], "gold": None, "text": None, "domain": None})

for model_name, df in dfs.items():
    for cid in common_ids:
        row = df.loc[cid]
        for label in row["fp_labels"]:
            entry = consensus_raw[(cid, label)]
            entry["models"].append(model_name)
            if entry["gold"] is None:
                entry["gold"]   = row["gold"]
                entry["text"]   = row["text"]
                entry["domain"] = row["domain"]

consensus_rows = []
for (cid, label), info in consensus_raw.items():
    n = len(info["models"])
    consensus_rows.append({
        "custom_id":          cid,
        "hallucinated_label": label,
        "n_models":           n,
        "models":             ", ".join(sorted(info["models"])),
        "gold_labels":        ", ".join(sorted(info["gold"])) if info["gold"] else "None",
        "domain":             info["domain"],
        "text":               info["text"],
    })

consensus_df = (
    pd.DataFrame(consensus_rows)
    .sort_values(["n_models", "hallucinated_label"], ascending=[False, True])
    .reset_index(drop=True)
)

print(f"Total (example, label) FP pairs: {len(consensus_df):,}")
print(f"  of which consensus (≥{N_CONSENSUS} models): {(consensus_df.n_models >= N_CONSENSUS).sum():,}")
print(f"  of which unanimous (all {len(dfs)} models):  {(consensus_df.n_models == len(dfs)).sum():,}")

In [ ]:
# Distribution of consensus FPs by label
consensus_strong = consensus_df[consensus_df["n_models"] >= N_CONSENSUS].copy()

label_dist = (
    consensus_strong
    .groupby("hallucinated_label")["n_models"]
    .agg(n_cases="count", avg_agreement="mean")
    .sort_values("n_cases", ascending=False)
)
label_dist["avg_agreement"] = label_dist["avg_agreement"].round(2)

print(f"Labels driving consensus FP cases (≥{N_CONSENSUS} models agree):")
display(label_dist)

In [ ]:
# Readable display of unanimous examples (all models agree) — top labels
# Re-run this cell to get a different random sample each time
N_MODELS_TOTAL = len(dfs)
unanimous = consensus_df[consensus_df["n_models"] == N_MODELS_TOTAL]

top_hallucinated = unanimous["hallucinated_label"].value_counts().head(5).index.tolist()

for label in top_hallucinated:
    sub = unanimous[unanimous["hallucinated_label"] == label]
    sample = sub.sample(min(3, len(sub)))
    print(f"\n{'='*70}")
    print(f"  LABEL: {label}  |  {len(sub)} unanimous FP cases (all {N_MODELS_TOTAL} models)")
    print(f"{'='*70}")
    for _, row in sample.iterrows():
        print(f"\n  [{row['domain']}]  id: {row['custom_id']}")
        print(f"  Gold labels : {row['gold_labels']}")
        print(f"  Text        : {row['text']}")

## 4. Model variance — which labels are model-specific?

For each label, compute the **coefficient of variation** (CV = std / mean) of FP counts across models.

CV normalizes by the label's average FP count, which matters here because labels differ a lot in magnitude
(Loaded Language ~1700 FPs/model vs Slogan ~260). A raw std of 200 means something very different for each.
CV answers: *is the spread large relative to how common this error is?*

- **High CV** → error concentrated in specific models (model-specific)  
- **Low CV** → all models err at roughly the same rate (systemic)

`std_fp` is shown alongside so you can judge absolute scale too.

In [ ]:
model_fp_data = label_fp_df.loc[top_labels, MODEL_NAMES]

variance_df = pd.DataFrame({
    "total_fp": model_fp_data.sum(axis=1),
    "mean_fp":  model_fp_data.mean(axis=1).round(1),
    "std_fp":   model_fp_data.std(axis=1).round(1),
    "cv":       (model_fp_data.std(axis=1) / (model_fp_data.mean(axis=1) + 1e-9)).round(3),
}).sort_values("cv", ascending=False)

print("Label-level cross-model variance (CV: high = model-specific, low = systemic):")
display(variance_df)

In [ ]:
# For the top-3 high-variance labels, show idiosyncratic examples (only 1 model hallucinates)
# Re-run this cell to get a different random sample each time
HIGH_VARIANCE_LABELS = variance_df.head(3).index.tolist()

for label in HIGH_VARIANCE_LABELS:
    idio = consensus_df[
        (consensus_df["hallucinated_label"] == label) &
        (consensus_df["n_models"] == 1)
    ]
    if idio.empty:
        print(f"\n{label}: no single-model FP cases found")
        continue

    model_breakdown = dict(Counter(idio["models"].tolist()))
    print(f"\n{'='*70}")
    print(f"  LABEL: {label}  (high cross-model variance)")
    print(f"  Idiosyncratic FP cases: {len(idio)}  |  By model: {model_breakdown}")
    print(f"{'='*70}")
    driving_models = sorted(model_breakdown, key=model_breakdown.get, reverse=True)[:3]
    for model in driving_models:
        sub = idio[idio["models"] == model]
        row = sub.sample(1).iloc[0]
        print(f"\n  [{row['domain']}]  Model: {model}")
        print(f"  Gold labels : {row['gold_labels']}")
        print(f"  Text        : {row['text']}")

## 5. Export examples for manual annotation

Saves a TSV of the most informative FP cases for qualitative reading.

In [ ]:
OUTPUT_TSV = Path("fp_error_analysis_examples.tsv")

# Re-run this cell to export a different random sample each time
export_unanimous = (
    unanimous[unanimous["hallucinated_label"].isin(top_hallucinated)]
    .assign(error_type="unanimous")
    .sample(min(30, len(unanimous[unanimous["hallucinated_label"].isin(top_hallucinated)])))
)

export_idio_rows = []
for label in HIGH_VARIANCE_LABELS:
    sub = consensus_df[
        (consensus_df["hallucinated_label"] == label) &
        (consensus_df["n_models"] == 1)
    ]
    export_idio_rows.append(sub.sample(min(5, len(sub))).assign(error_type="idiosyncratic"))

export_df = pd.concat([export_unanimous] + export_idio_rows, ignore_index=True)
export_df.to_csv(OUTPUT_TSV, sep="\t", index=False)
print(f"Saved {len(export_df)} examples to {OUTPUT_TSV}")